# Does the I-JEPA world model rank candidate actions correctly?

This notebook compares **world-model predicted goal-latent MSE** with **real goal-latent MSE** after executing the same candidate actions in the Wall simulator. It uses the multi-layout world model and the matching I-JEPA **1K** encoder. No episode dataset or latent cache is needed.

Run the cells in order in a **Colab GPU runtime**. The default experiment uses the seen layout `(wall x=32, door y=30)`, three fresh start–goal cases, horizons 1 and 5, and the training action-length cap of 3.5. Results are saved to Google Drive.


## 1. Mount Drive, clone the code, and install dependencies

If you rerun this cell in the same runtime, it updates the I-JEPA branch and reuses the existing clones.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import subprocess
import sys

REPO = Path("/content/i-jepa-wm")
DINO_ROOT = Path("/content/dino_wm")
DINO_COMMIT = "0a9492fa12044b852ae9e001cc74604b79c8bb0c"

def run(command):
    print("$", " ".join(map(str, command)), flush=True)
    subprocess.run(command, check=True)

if not REPO.exists():
    run([
        "git", "clone", "--branch", "wall-layout-generalization",
        "--single-branch", "https://github.com/kmylonas/i-jepa-wm.git", str(REPO),
    ])
else:
    run(["git", "-C", str(REPO), "fetch", "origin", "wall-layout-generalization"])
    run(["git", "-C", str(REPO), "switch", "wall-layout-generalization"])
    run(["git", "-C", str(REPO), "pull", "--ff-only"])

if not DINO_ROOT.exists():
    run(["git", "clone", "https://github.com/gaoyuezhou/dino_wm.git", str(DINO_ROOT)])
run(["git", "-C", str(DINO_ROOT), "checkout", DINO_COMMIT])

run([
    sys.executable, "-m", "pip", "install", "-q",
    "gym==0.26.2", "omegaconf", "scipy", "pyyaml",
    "matplotlib", "imageio", "imageio-ffmpeg",
])

import torch
assert torch.cuda.is_available(), "Select a GPU runtime in Colab first"
assert (REPO / "src/mylonas_ijepa_wm/evaluate_wall_candidate_rankings.py").is_file()
print("GPU:", torch.cuda.get_device_name(0))
print("I-JEPA commit:", subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"], text=True
).strip())


## 2. Locate the saved checkpoints and prepare the environment config

This reads the trained multi-layout model and encoder from Drive. It copies only the encoder checkpoint to the local runtime for faster loading. The experiment outputs stay on Drive.


In [ ]:
import json
import shutil

DRIVE_ROOT = Path("/content/drive/MyDrive/datasets/wall_layout_generalization")
CHECKPOINT = DRIVE_ROOT / "world_model/best.pt"
ENCODER_DRIVE = Path(
    "/content/drive/MyDrive/checkpoints/"
    "IN1K-vit.h.14-300e.encoder-only.pth.tar"
)
ENCODER_LOCAL = Path("/content/ijepa_1k_encoder.pth.tar")
ENV_CONFIG = DRIVE_ROOT / "environment.json"
LOCAL_CONFIG = Path("/content/wall_candidate_rankings_environment.json")
OUTPUT_DIR = DRIVE_ROOT / "results/candidate_rankings_wall32_door30_v1"

for path in (
    REPO / "src/mylonas_ijepa_wm/evaluate_wall_candidate_rankings.py",
    DINO_ROOT / "env/wall/envs/wall.py",
    CHECKPOINT, ENCODER_DRIVE, ENV_CONFIG,
):
    assert path.is_file(), f"Missing: {path}"

if not ENCODER_LOCAL.is_file():
    shutil.copy2(ENCODER_DRIVE, ENCODER_LOCAL)
settings = json.loads(ENV_CONFIG.read_text())
settings["dino_wm_root"] = str(DINO_ROOT)
LOCAL_CONFIG.write_text(json.dumps(settings, indent=2))

print("World model:", CHECKPOINT)
print("Encoder:", ENCODER_LOCAL)
print("Results:", OUTPUT_DIR)


## 3. Run the candidate replay

For each case and horizon, CEM scores candidate action sequences with the world model. The script then resets the simulator before executing each selected sequence and encodes its real terminal frame. It prints progress after each of the six case/horizon combinations.


In [ ]:
command = [
    sys.executable, "-u", "-m",
    "src.mylonas_ijepa_wm.evaluate_wall_candidate_rankings",
    "--world-model-checkpoint", str(CHECKPOINT),
    "--encoder-checkpoint", str(ENCODER_LOCAL),
    "--env-config", str(LOCAL_CONFIG),
    "--output-dir", str(OUTPUT_DIR),
    "--wall-x", "32", "--door-y", "30",
    "--device", "cuda", "--precision", "auto",
]
print("Running candidate replay; outputs will be saved to", OUTPUT_DIR, flush=True)
subprocess.run(command, cwd=REPO, check=True)


## 4. Read the ranking metrics

The most useful number is **predicted/real ρ** for `final_elite`: a high value means the world model ranks the candidates CEM favors similarly to their real endpoint costs. **True/distance ρ** checks whether the encoder's real-image cost tracks straight-line goal distance. A wall may require a detour, so distance is supporting context. Costs from the I-JEPA encoder are only compared with other costs from that same encoder.

**Selection-rule note:** these figures contain scored *sampled* candidates. New latent-cost planning returns the mean of the final elites, which is a separate sequence. The `predicted-best` numbers below describe the old best-sampled rule; this notebook does not replay the returned elite mean.


In [ ]:
results = json.loads((OUTPUT_DIR / "summary.json").read_text())

def fmt_rho(value):
    return "undefined" if value is None else f"{value:.3f}"

for case in results["cases"]:
    print(f"seed {case['seed']}, horizon {case['horizon']}")
    for group in ("initial_random", "final_elite", "all"):
        metrics = case["rankings"][group]
        print(
            f"  {group:14s}  "
            f"predicted/real ρ={fmt_rho(metrics['predicted_vs_true_rho'])}  "
            f"true/distance ρ={fmt_rho(metrics['true_vs_distance_rho'])}  "
            f"predicted-best real cost={metrics['predicted_best_true_cost']:.4f}  "
            f"best available real cost={metrics['oracle_best_true_cost']:.4f}"
        )
    print()


## 5. View the figures

Each figure shows predicted versus real endpoint latent MSE on the left, and real latent MSE versus true endpoint distance on the right. Gray points are sampled from the first CEM round; orange points are the best candidates from the final round. The complete per-candidate data is also saved as `candidates.csv` alongside the figures and `summary.json`.


In [ ]:
from IPython.display import Image, display

for case in results["cases"]:
    figure = OUTPUT_DIR / f"seed_{case['seed']}_h{case['horizon']}.png"
    print(f"seed {case['seed']}, horizon {case['horizon']}")
    display(Image(filename=str(figure)))
